![banner](../../../docs/figs/brand_identity/banner.png)

# LLM: Document Data Extraction & Processing 

![banner](../../../docs/figs/multimodal_llm/extraction.png)

## Tools

#### A. LLM

In [1]:
from ccai9012 import llm_utils
import pandas as pd
import os

C:\Users\Joseph\anaconda3\envs\ccai9012\Lib\site-packages\diffusers\models\transformers\transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
C:\Users\Joseph\anaconda3\envs\ccai9012\Lib\site-packages\diffusers\models\transformers\transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


In [3]:
# Initialize LLM
api_key = llm_utils.get_deepseek_api_key() #LLM from DeepSeek
llm = llm_utils.initialize_llm()

In [4]:
# Test
prompt = ["Is 9.9 or 9.11 bigger?"]
llm_utils.ask_llm(prompt)


📌 Prompt:
['Is 9.9 or 9.11 bigger?']

Let's compare the two numbers:  

**9.9** means 9 and 9 tenths, or \( 9.90 \) if we write it with two decimal places.  
**9.11** means 9 and 11 hundredths, or \( 9.11 \).  

Comparing digit by digit:  
- Ones place: both are **9**.  
- Tenths place: 9.9 has **9** in the tenths place, while 9.11 has **1** in the tenths place.  

Since \( 9 > 1 \) in the tenths place, **9.9 is larger** than 9.11.  

\[
9.9 > 9.11
\]  

**Answer:** 9.9 is bigger.



#### B. Retrieval-Augmented Generation (RAG)
1. sparse and embedding (extracting text and turn them to numbers)
2. retrieval (search mathematically similar text chunks)
3. generation (augmenting an answer)

In [5]:
retriever = llm_utils.build_pdf_retriever(
    pdf_path="data/1526439.pdf",
    embedding_model_name="BAAI/bge-base-en-v1.5" #RAG from huggingface
)

C:\Users\Joseph\ccai9012\ccai9012\llm_utils.py:639: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name=embedding_model_name)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Lets combine them!

#### Summarizing 1 document
QA Chain: Retriever + Prompt Template + LLM (can search)

In [5]:
summary = llm_utils.run_qa_chain( #function that retrieves most relevant snippets from retriever, and use them as prompt input for LLM
    query="Please summarize the location, main objectives, actions, and stakeholders described in this energy plan document.",
    retriever=retriever,
    llm=llm,
    return_sources=True,  #optional (verify sources)
    save_path="output/summary.txt"
)


--- Final Answer ---
Based on the provided context, here is a summary of the location, main objectives, actions, and stakeholders of the energy plan document.

**Location:** The plan is for the community of **Aniak, Alaska**.

**Main Objectives:**
1.  To create an **Energy Action Plan** for tribal buildings in Aniak.
2.  To make the buildings **safer, more comfortable, and more energy efficient**.
3.  To provide a starting point and guide for future building retrofit projects.

**Key Actions:**
The project followed a multi-step procedure to complete the plan:
1.  **Building Selection & Goal Setting:** The Aniak Traditional Council selected seven tribal buildings and set goals for each.
2.  **Kick-off Meeting:** Project staff and the Council met to discuss goals, procedures, and resources.
3.  **Data Collection:** Gathering baseline information on building energy use, occupancy, and condition.
4.  **Energy Audits:** On-site assessments of each building were conducted to provide priorit

## More fun!

#### Add system prompt

In [6]:
from langchain.prompts import PromptTemplate

structured_prompt = PromptTemplate.from_template(
"""
Given the following document text, extract key information and output a markdown table with columns:

| Location | Main Objectives | Key Actions | Stakeholders | Timeline |
|----------|-----------------|-------------|--------------|----------|

Use exact information from the text; if any info is missing, write "N/A".

Context:
{context}

Question:
{question}

"""
)

#### Summarizing 2+ document

In [7]:
folder_path = "data"
save_csv_path = "output/multiple_comparison.csv"
query_text = "Extract key information from this document."
results = [] #initialize list container for responses

for fname in sorted(os.listdir(folder_path)): # list out all files in directory, loop through every pdf
    if not fname.lower().endswith(".pdf"): # if its not a pdf, ignore and continue
        continue
    pdf_path = os.path.join(folder_path, fname) #f ull file path, joining file path and name 
    print(f"Processing {fname} ...")

    # Build RAG retriever for this pdf
    retriever = llm_utils.build_pdf_retriever(pdf_path)

    # Run QA chain for extraction
    extracted_text = llm_utils.run_qa_chain(
        query=query_text,
        retriever=retriever,
        llm=llm,
        prompt_template=structured_prompt, # added structured prompt
        return_sources=False
    )

    results.append({  # creating dictionary for dataframe
        "pdf_path": fname,
        "extracted_text": extracted_text,
    })

df = pd.DataFrame(results)
os.makedirs(os.path.dirname(save_csv_path), exist_ok=True)
df.to_csv(save_csv_path, index=False, encoding="utf-8-sig") #translation settings to csv
print(f"\n Saved results to {save_csv_path}") # save to csv

Processing 1526439.pdf ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Final Answer ---
Based on the provided text, here is the extracted key information in the requested markdown table format.

| Location | Main Objectives | Key Actions | Stakeholders | Timeline |
| :--- | :--- | :--- | :--- | :--- |
| Aniak, Alaska | 1. Improve building condition.<br>2. Decrease energy use.<br>3. Develop a thorough and complete project plan for retrofits.<br>4. Showcase project objectives and results to the community. | 1. Survey buildings (layout, insulation, system condition, safety).<br>2. Conduct building energy audits using AkWarm-C software.<br>3. Develop an Energy Action Plan (draft, present, revise, finalize).<br>4. Develop a data monitoring plan.<br>5. Collect baseline data (energy use, condition, comfort).<br>6. Create a monthly maintenance checklist.<br>7. Identify financing options for retrofits.<br>8. Conduct outreach activities. | • Cold Climate Housing Research Center (CCHRC)<br>• Energy Audits of Alaska<br>• Aniak Traditional Council / The Tribe<br>

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Final Answer ---
Based on the provided text, here is the extracted information in the requested markdown table format.

| Location | Main Objectives | Key Actions | Stakeholders | Timeline |
| :--- | :--- | :--- | :--- | :--- |
| N/A | Showcase the objective and results of the project to community members, Alaskans, and others. | Conduct outreach activities; store project documents (flyers, scope of work, audit summaries) in appendices. | Community members, Alaskans, others | N/A |
| Tribal office file cabinet; CCHRC server/website | Store project data for reference and future use. | Store hard copies of fuel/electrical data; store building condition/occupant comfort data (if collected); store baseline data on CCHRC server and website. | The Tribe, Tribal administrator, Cold Climate Housing Research Center (CCHRC) | N/A |
| Police station; ATC shop | Perform regular building maintenance and inspections. | **Police Station:** Check/configure thermostat; create heating parts list; e

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Final Answer ---
Based on the provided document text, here is the extracted key information in the requested markdown table format.

| Location | Main Objectives | Key Actions | Stakeholders | Timeline |
|----------|-----------------|-------------|--------------|----------|
| Akiachak (Tribal buildings) | 1. Conduct energy audits of tribal buildings. <br> 2. Develop an Energy Action Plan. <br> 3. Improve energy efficiency. <br> 4. Track building improvements. | 1. Meet to discuss project goals and buildings. <br> 2. Collect baseline data (fuel oil use, electrical use, occupant comfort, building condition). <br> 3. Write a data monitoring plan for each building. <br> 4. Complete on-site building assessments. <br> 5. Prepare a draft Energy Action Plan. <br> 6. Present draft plan and gather feedback. <br> 7. Revise and finalize the plan. <br> 8. Provide final plan to the Tribe. <br> 9. Store data (hard copies in tribal office, project folder on CCHRC server). | Cold Climate Housing R

#### Combining the results into one file

In [9]:
merged_df_list = []
df = pd.read_csv("output/multiple_comparison.csv")

for i, md in enumerate(df["extracted_text"]): # extracting with labels
    try:
        parsed = llm_utils.parse_markdown_table(md) # translate (|) and (-) to table
        parsed["source_doc"] = df.loc[i, "pdf_path"] if "pdf_path" in df.columns else f"doc_{i+1}" # adding new label to mark file source
        merged_df_list.append(parsed) # joining as list
    except Exception as e:
        print(f"Failed to parse doc {i+1}: {e}")

final_df = pd.concat(merged_df_list, ignore_index=True) # joined listto dataframe
final_df.to_csv("output/merged_energy_policy.csv", index=False) # save as csv

In [10]:
final_df

,Location,Main Objectives,Key Actions,Stakeholders,Timeline,source_doc
0,"Aniak, Alaska",1. Improve building condition.; 2. Decrease en...,"1. Survey buildings (layout, insulation, syste...",• Cold Climate Housing Research Center (CCHRC)...,N/A,1526439.pdf
1,Kwigillingok,1. Improve building condition. ; 2. Decrease ...,"1. Survey buildings (layout, insulation, occup...",1. Tribal finance officer (data oversight). ; ...,N/A,1526994.pdf
2,N/A,Showcase the objective and results of the proj...,Conduct outreach activities; store project doc...,"Community members, Alaskans, others",N/A,1527001.pdf
3,Tribal office file cabinet; CCHRC server/website,Store project data for reference and future use.,Store hard copies of fuel/electrical data; sto...,"The Tribe, Tribal administrator, Cold Climate ...",N/A,1527001.pdf
4,Police station; ATC shop,Perform regular building maintenance and inspe...,**Police Station:** Check/configure thermostat...,"Police station staff, ATC shop staff",Monthly (for ATC shop exterior inspection),1527001.pdf
5,N/A,Develop a strong project plan for funding appl...,"Use audit info to define scope, finances, time...","Project applicants, other communities",N/A,1527001.pdf
6,N/A,Plan and initiate the energy audit project.,Hold kickoff meeting to decide metrics; develo...,"Council, Project staff, Tribal office staff, B...",N/A,1527001.pdf
7,N/A,Address key questions for project planning.,Identify local retrofit contractors; determine...,"Native Village of Atmautluak, CCHRC",N/A,1527001.pdf
8,N/A,Complete the Energy Action Plan.,1. Initial project meeting.; 2. Collect baseli...,"CCHRC, Energy Audits of Alaska, The Tribe, Bui...",N/A,1527001.pdf
9,N/A,Identify funding and training opportunities fo...,"Provide a comprehensive, color-coded table of ...","Applicants, Alaska Native Health Consortium, A...",At time of publication (2019),1527001.pdf


## Further summary?
LLMChain: Prompt Template + LLM (no search)

In [19]:
summary_prompt = PromptTemplate.from_template(
    """
    Below are extracted policy tables from multiple documents.

    Please compare and summarize:
    - What are the common energy policy goals?
    - Which regions have the most comprehensive plans?
    - Are there unique or innovative actions mentioned?
    - Summarize the main differences in stakeholders and timelines.

    Keep the answer concise and structured in paragraphs with bullet points.

    Tables:
    {context}
    """
)

from langchain.chains import LLMChain

all_tables = "\n\n".join(df["extracted_text"].dropna().tolist()) # join to list separated with blank line

llm_chain = LLMChain(prompt=summary_prompt, llm=llm)
summary = llm_chain.invoke({"context": all_tables}) #adding information to prompt

with open("output/energy_policy_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary["text"])

print(summary["text"]) #'text' is a key that LangChain uses to label the final answer

**Common Energy Policy Goals:**
The primary goals across all communities are consistent: to improve building conditions, decrease energy use, and develop structured plans (Energy Action Plans) to facilitate future energy efficiency retrofits. A strong secondary goal is community engagement and data sharing to showcase results and inform others.

**Comprehensiveness of Plans:**
The plans for **Aniak, Kwigillingok, and Akiachak** are the most comprehensive, as they follow a complete, documented process from initial audit to final plan. They include detailed steps for data collection, analysis, stakeholder engagement, and maintenance. The other entries are more focused on specific sub-tasks or general guidance.

**Unique or Innovative Actions:**
*   The development of a **color-coded funding opportunity table** (grants, loans, technical assistance) is a notable innovative tool to directly address implementation barriers.
*   Emphasis on creating **formal data monitoring plans** and dual s

## Other use cases

![banner](../../../docs/figs/multimodal_llm/otherusecase.png)